# Contextual Compression in Document Retrieval

## Overview

Standard retrieval returns entire chunks — but often only a small part of the chunk is actually relevant to the query. **Contextual compression** uses an LLM to extract only the relevant parts from each retrieved chunk.

| Standard Retrieval | Contextual Compression |
|---|---|
| Returns full chunks (may contain irrelevant text) | Returns only the **relevant parts** of each chunk |
| LLM must sift through noise | LLM gets focused, compressed context |

## How It Works

1. Retrieve chunks via standard vector search
2. For each chunk, ask the LLM: *"Extract only the parts relevant to the question"*
3. Use the compressed results as context

## Models Used

- **LLM**: `gemma3:12b` via Ollama (contextual compression + answer generation)
- **Embeddings**: `mxbai-embed-large:335m` via Ollama

<div style="text-align: center;">

<img src="./images/contextual_compression.svg" alt="Contextual Compression" style="width:70%; height:auto;">
</div>

---
## Step 0: Import Packages

In [ ]:
from langchain_ollama import ChatOllama
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

---
## Step 1: Set Up LLM and Embedding Model

In [ ]:
llm = ChatOllama(temperature=0, model="gemma3:12b", max_tokens=4000)
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")

print("LLM and embedding model ready")

---
## Step 2: Load PDF, Chunk, Build Vector Store

In [ ]:
path = "data/Understanding_Climate_Change.pdf"

loader = PyPDFLoader(path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, length_function=len
)
chunks = text_splitter.split_documents(documents)

vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever()

print(f"Loaded {len(documents)} pages, split into {len(chunks)} chunks")
print(f"Vector store and retriever created")

---
## Step 3: Define the Compression Prompt

This prompt instructs the LLM to extract only the parts of the chunk that are relevant to the question.

In [ ]:
compression_prompt = ChatPromptTemplate.from_template(
    "Given the following context, extract only the parts that are relevant to the question.\n\n"
    "Context: {context}\n\n"
    "Question: {question}\n\n"
    "Relevant parts:"
)

compression_chain = compression_prompt | llm | StrOutputParser()

print("Compression chain ready")

---
## Step 4: Retrieve and Compress

First, retrieve chunks normally. Then, for each chunk, use the LLM to extract only the relevant parts.

In [ ]:
query = "What is the main topic of the document?"
print(f"Query: {query}\n")

# Step A: Standard retrieval
retrieved_docs = retriever.invoke(query)
print(f"Retrieved {len(retrieved_docs)} chunks\n")

# Show what we got before compression
print("=== Before Compression (first chunk) ===")
print(retrieved_docs[0].page_content[:300] + "...")
print(f"Length: {len(retrieved_docs[0].page_content)} chars")

In [ ]:
# Step B: Compress each chunk
compressed_results = []
for i, doc in enumerate(retrieved_docs):
    result = compression_chain.invoke({
        "context": doc.page_content,
        "question": query
    })
    if result.strip():
        compressed_results.append(result.strip())
        print(f"Chunk {i+1}: {len(doc.page_content)} chars -> {len(result.strip())} chars (compressed)")

print(f"\nCompressed {len(retrieved_docs)} chunks into {len(compressed_results)} results")

---
## Step 5: View the Compressed Results

In [ ]:
for i, result in enumerate(compressed_results):
    print(f"\n--- Compressed Chunk {i+1} ---")
    print(result)

---
## Step 6: Use Compressed Context to Answer the Question

In [ ]:
combined_context = "\n\n".join(compressed_results)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant for question-answering tasks. Use the following context to answer the question. "
     "Keep the answer concise (3-5 sentences)."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

answer_chain = answer_prompt | llm | StrOutputParser()
answer = answer_chain.invoke({"context": combined_context, "question": query})

print(f"Question: {query}\n")
print(f"Answer: {answer}")

---
## Summary

| Step | What happened |
|---|---|
| 1 | Set up LLM + embeddings |
| 2 | Loaded PDF, chunked, built FAISS vector store |
| 3 | Defined compression prompt ("extract only relevant parts") |
| 4 | **Retrieved chunks** then **compressed each one** with the LLM |
| 5 | Viewed compressed results — much shorter and more focused |
| 6 | Used compressed context to generate a final answer |

**Key insight:** Contextual compression acts as a precision filter between retrieval and generation. Instead of feeding the LLM entire 1000-character chunks (which may be 80% irrelevant), we compress each chunk to just the relevant sentences. This means the LLM works with cleaner input and produces more focused answers. The trade-off is an extra LLM call per chunk, but for high-precision applications it's worth it.